In [1]:
# =============================================================================
# Vector Database
# =============================================================================
import chromadb

# =============================================================================
# Data Processing
# =============================================================================
import pandas as pd
import numpy as np

In [2]:
# =============================================================================
# Load Semantic Chunk Metadata
# =============================================================================
semantic_dataset = pd.read_csv("semantic_chunks.csv")
print("Semantic Dataset Shape:", semantic_dataset.shape)

semantic_dataset.head()

Semantic Dataset Shape: (113871, 9)


,chunk_id,QuestionID,Category,QuestionType,QuestionTime,chunk_index,chunk_text,search_text,embedding
0,row_0_chunk_0,C15Q2112,Tools and Home Improvement,open-ended,2013-04-20,0,Question: - what are the dimensions of this it...,Category: Tools and Home Improvement Question ...,"[0.005750724114477634, 0.0893775224685669, -0...."
1,row_1_chunk_0,C9Q4595,Home and Kitchen,open-ended,2014-02-06,0,Question: how much booze can it hold? Answer: ...,Category: Home and Kitchen Question Type: open...,"[0.10887277871370316, 0.08158908039331436, -0...."
2,row_2_chunk_0,C4Q7999,Cell Phones and Accessories,open-ended,2014-08-09,0,Question: will this case fit nokia lumia 520 A...,Category: Cell Phones and Accessories Question...,"[-0.11016276478767395, 0.046286050230264664, 0..."
3,row_3_chunk_0,C8Q8916,Health and Personal Care,open-ended,2014-04-25,0,"Question: when folded in the sitting position,...",Category: Health and Personal Care Question Ty...,"[0.0285594891756773, 0.017597414553165436, 0.0..."
4,row_4_chunk_0,C14Q905,Sports and Outdoors,open-ended,2015-04-15,0,Question: how long should i leave this on to g...,Category: Sports and Outdoors Question Type: o...,"[-0.0072077070362865925, 0.05245823785662651, ..."


In [3]:
# =============================================================================
# Load Pre-generated Embeddings
# =============================================================================
embeddings = np.load( "semantic_embeddings.npy")
print("Embedding Matrix Shape:", embeddings.shape)

Embedding Matrix Shape: (113871, 384)


In [4]:
# =============================================================================
# Validate Data Alignment
# =============================================================================
assert len(semantic_dataset) == len(embeddings)
print("Chunks and embeddings are aligned successfully.")

Chunks and embeddings are aligned successfully.


In [5]:
# =============================================================================
# Create Persistent Chroma Database
# =============================================================================
client = chromadb.PersistentClient( path="./chroma_db")
print("Chroma client created successfully.")

Chroma client created successfully.


In [6]:
# =============================================================================
# Create Semantic Search Collection
# =============================================================================
collection = client.get_or_create_collection( name ="amazon_qa_semantic", metadata={"hnsw:space": "cosine"})

In [7]:
# =============================================================================
# Prepare Metadata for Chroma
# =============================================================================
metadatas = []

for _, row in semantic_dataset.iterrows():

    metadatas.append({

        "QuestionID": str(row["QuestionID"]),

        "Category": str(row["Category"]),

        "QuestionType": str(row["QuestionType"]),

        "QuestionTime": str(row["QuestionTime"])})

print("Metadata prepared successfully.")

Metadata prepared successfully.


In [8]:
# Check Duplicates
print(semantic_dataset[["chunk_id"]].head(10))

        chunk_id
0  row_0_chunk_0
1  row_1_chunk_0
2  row_2_chunk_0
3  row_3_chunk_0
4  row_4_chunk_0
5  row_5_chunk_0
6  row_6_chunk_0
7  row_7_chunk_0
8  row_8_chunk_0
9  row_9_chunk_0


In [9]:
# Check Duplicates
print("Rows:", len(semantic_dataset))
print("Unique IDs:", semantic_dataset["chunk_id"].nunique())

Rows: 113871
Unique IDs: 113871


In [10]:
# Check Duplicates
duplicates = semantic_dataset[ semantic_dataset["chunk_id"].duplicated(keep=False)].sort_values("chunk_id")

duplicates[ ["QuestionID", "chunk_id", "chunk_text"]].head(20)

,QuestionID,chunk_id,chunk_text


In [12]:
# =============================================================================
# Store Embeddings in Batches
# =============================================================================
batch_size = 5000

for start in range(0, len(semantic_dataset), batch_size):

    end = start + batch_size

    collection.add(
        ids=semantic_dataset["chunk_id"].iloc[start:end].astype(str).tolist(),
        documents=semantic_dataset["chunk_text"].iloc[start:end].fillna("").tolist(),
        embeddings=embeddings[start:end].tolist(),
        metadatas=metadatas[start:end], )

    print(f"Stored {end} chunks")

Stored 5000 chunks
Stored 10000 chunks
Stored 15000 chunks
Stored 20000 chunks
Stored 25000 chunks
Stored 30000 chunks
Stored 35000 chunks
Stored 40000 chunks
Stored 45000 chunks
Stored 50000 chunks
Stored 55000 chunks
Stored 60000 chunks
Stored 65000 chunks
Stored 70000 chunks
Stored 75000 chunks
Stored 80000 chunks
Stored 85000 chunks
Stored 90000 chunks
Stored 95000 chunks
Stored 100000 chunks
Stored 105000 chunks
Stored 110000 chunks
Stored 115000 chunks


In [13]:
print(f"Stored {min(end, len(semantic_dataset))} / {len(semantic_dataset)}")

Stored 113871 / 113871


In [14]:
# =============================================================================
# Test Semantic Retrieval
# =============================================================================
query_embedding = embeddings[0].tolist()

results = collection.query(query_embeddings =[query_embedding], n_results=3)
results

{'ids': [['row_0_chunk_0', 'row_20046_chunk_0', 'row_76247_chunk_0']],
 'embeddings': None,
 'documents': [['Question: - what are the dimensions of this item? Answer:',
   'Question: - what are the dimensions of this item? Answer: two keys',
   'Question: what are the dimensions? Answer: 5 1/2']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'QuestionID': 'C15Q2112',
    'QuestionTime': '2013-04-20',
    'QuestionType': 'open-ended',
    'Category': 'Tools and Home Improvement'},
   {'QuestionType': 'open-ended',
    'QuestionTime': '2013-04-20',
    'QuestionID': 'C15Q2112',
    'Category': 'Tools and Home Improvement'},
   {'Category': 'Tools and Home Improvement',
    'QuestionTime': '2013-07-14',
    'QuestionID': 'C15Q582',
    'QuestionType': 'open-ended'}]],
 'distances': [[0.0, 0.08064407110214233, 0.14212243258953094]]}

In [15]:
# ==========================================================
# Verify Chroma Database Persistence
# ==========================================================
print("Number of stored documents:",collection.count())
print( "Chroma database saved successfully.")

Number of stored documents: 113871
Chroma database saved successfully.
